## <Smithsonian(스미소니언 협회) 데이터 수집>
- API 설명: https://edan.si.edu/openaccess/apidocs/
- API 키 발급: https://api.data.gov/signup (환경변수 `SI_API_KEY`로 설정)

In [1]:
!pip install pillow

### 1. 필요한 것들 불러오기

In [2]:
import os
import time
import requests
import pandas as pd

MUSEUM_CODE = "SI"
OUTPUT_EXCEL = f"../data/{MUSEUM_CODE}_hyungbae.xlsx"
HEADERS = {"User-Agent": "aks-digital-humanities-research/1.0"}
API_KEY = os.environ.get("SI_API_KEY")

### 2. API 호출해서 원하는 정보 얻기

In [4]:
CATEGORY = "흉배"
CATEGORY_LETTER = "H"
KEYWORD = "rank badge"

# 스미소니언 산하 기관 코드 → 한글명
UNIT_NAMES = {
    "CHNDM": "스미소니언 쿠퍼 휴잇 디자인 박물관",
    "FSG": "스미소니언 국립아시아미술관",
    "NMAA": "스미소니언 국립아시아미술관",
    "NMNH": "스미소니언 국립자연사박물관",
    "NMNHANTHRO": "스미소니언 국립자연사박물관",
    "NMAH": "스미소니언 국립미국역사박물관",
}

rows = []
serial_counter = {}
failed_ids = []

def next_temp_id(museum_code):
    n = serial_counter.get(museum_code, 0) + 1
    serial_counter[museum_code] = n
    return f"Y{museum_code}{CATEGORY_LETTER}{n:02d}"

def search_si(keyword=KEYWORD, page_size=100):
    base = "https://api.si.edu/openaccess/api/v1.0/search"
    start = 0
    found = []
    while True:
        params = {"q": f'"{keyword}"', "start": start, "rows": page_size, "api_key": API_KEY}
        resp = requests.get(base, params=params, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        payload = resp.json().get("response") or {}
        records = payload.get("rows") or []
        if not records:
            break
        for rec in records:
            title = rec.get("title") or ""
            if keyword.lower() in title.lower():   # 검색 단계에서 바로 필터링 (제목 기준)
                found.append(rec["id"])

        start += page_size
        if start >= (payload.get("rowCount") or 0):
            break
        time.sleep(0.3)
    return list(dict.fromkeys(found))  # 중복 제거, 순서 유지

def fetch_si_detail(record_id):
    url = f"https://api.si.edu/openaccess/api/v1.0/content/{record_id}"
    resp = requests.get(url, params={"api_key": API_KEY}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json().get("response") or {}

# freetext 필드는 [{"label": ..., "content": ...}] 형태
def ft(content, key, labels=None, exclude=None):
    out = []
    for x in (content.get("freetext") or {}).get(key) or []:
        label = (x.get("label") or "").lower()
        if labels and not any(l in label for l in labels):
            continue
        if exclude and any(l in label for l in exclude):
            continue
        if x.get("content"):
            out.append(x["content"])
    return "; ".join(out)

def fetch_si(keyword=KEYWORD):
    record_ids = search_si(keyword)
    print(f"제목 기준 필터 후 대상: {len(record_ids)}건")

    for rid in record_ids:
        time.sleep(0.3)
        try:
            item = fetch_si_detail(rid)
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"{rid} 실패: {e}")
            failed_ids.append(rid)
            continue

        temp_id = next_temp_id(MUSEUM_CODE)
        content = item.get("content") or {}
        dnr = content.get("descriptiveNonRepeating") or {}
        indexed = content.get("indexedStructured") or {}
        unit = item.get("unitCode") or dnr.get("unit_code") or ""

        media = [m for m in ((dnr.get("online_media") or {}).get("media") or []) if m.get("type") == "Images"]
        image_urls = [m["content"] for m in media if m.get("content")]
        licenses = sorted({(m.get("usage") or {}).get("access", "") for m in media} - {""})

        accession = ft(content, "identifier", labels=["id number", "accession", "catalog"]) or ft(content, "identifier")

        rows.append({
            "임시ID": temp_id,
            "분류": CATEGORY,
            "소장처": UNIT_NAMES.get(unit, f"스미소니언 협회 ({unit})"),
            "소장처유물번호": accession,
            "한글명": "",
            "한자명": "",
            "영어명": item.get("title"),
            "URL": dnr.get("record_link") or dnr.get("guid"),
            "수집자": "전윤우",
            "searched_keyword": keyword,
            "objectType": "; ".join(indexed.get("object_type") or []) or ft(content, "objectType"),
            "place": ft(content, "place"),
            "date": ft(content, "date"),
            "materials": ft(content, "physicalDescription", labels=["medium", "material"]),
            "techniques": ft(content, "physicalDescription", labels=["technique"]),
            "dimensions": ft(content, "physicalDescription", labels=["dimension"]),
            "briefDescription": ft(content, "notes", labels=["description", "brief"]),
            "summaryDescription": ft(content, "notes", exclude=["description", "brief", "provenance", "history"]),
            "physicalDescription": ft(content, "physicalDescription", exclude=["medium", "material", "technique", "dimension"]),
            "objectHistory": ft(content, "notes", labels=["provenance", "history"]) or ft(content, "provenance"),
            "creditLine": ft(content, "creditLine"),
            "image_url": image_urls[0] if image_urls else None,
            "image_urls": "; ".join(image_urls),
            "license_note": f"Smithsonian ({', '.join(licenses)})" if licenses else "Smithsonian (이미지 없음)",
            "unitCode": unit,
            "si_id": rid,
        })

    if failed_ids:
        print(f"총 {len(failed_ids)}건 실패: {failed_ids}")

fetch_si()
print(f"{len(rows)}건 수집 완료")

HTTPError: 403 Client Error: Forbidden for url: https://api.si.edu/openaccess/api/v1.0/search?q=%22rank+badge%22&start=0&rows=100

### 3. 데이터프레임 -> 엑셀

In [ ]:
df = pd.DataFrame(rows)
master_cols = ["임시ID", "분류", "소장처", "소장처유물번호", "한글명", "한자명", "영어명",
               "URL", "searched_keyword", "objectType", "place", "date", "materials", "techniques",
               "dimensions", "briefDescription", "summaryDescription", "physicalDescription",
               "objectHistory", "creditLine", "image_url", "image_urls", "license_note"]
other_cols = [c for c in df.columns if c not in master_cols]
df = df[master_cols + other_cols]

pure_hyungbae = df[df["영어명"].str.contains("rank badge", case=False, na=False)]

os.makedirs("../data", exist_ok=True)
pure_hyungbae.to_excel(OUTPUT_EXCEL, index=False)

print(pure_hyungbae.shape)
pure_hyungbae.head()

### 4. 이미지 수집

In [ ]:
from PIL import Image

def download_si_images(df, image_root="../image", max_px=5000):
    folder_map = {"흉배": "hyungbae", "단령": "dallyeong", "초상": "portrait"}
    downloaded, skipped, failed, sizes = 0, 0, [], []

    for _, row in df.iterrows():
        folder = folder_map.get(row["분류"])
        if folder is None:
            continue
        folder_path = os.path.join(image_root, folder)
        os.makedirs(folder_path, exist_ok=True)

        urls = str(row["image_urls"]).split("; ") if pd.notna(row.get("image_urls")) and row.get("image_urls") else []
        if not urls:
            skipped += 1
            continue

        for i, url in enumerate(urls, start=1):
            suffix = "" if len(urls) == 1 else f"-{i}"
            save_path = os.path.join(folder_path, f"{row['임시ID']}{suffix}.jpg")
            if "ids.si.edu" in url:  # IDS 주소는 max로 최대 크기 요청 (원본 링크는 그대로)
                url = f"{url}{'&' if '?' in url else '?'}max={max_px}"
            try:
                resp = requests.get(url, headers=HEADERS, timeout=60)
                resp.raise_for_status()
                with open(save_path, "wb") as f:
                    f.write(resp.content)
                w, h = Image.open(save_path).size
                sizes.append((row["임시ID"] + suffix, w, h))
                downloaded += 1
            except requests.exceptions.RequestException as e:
                failed.append((row["임시ID"], str(e)))
            time.sleep(0.2)

    print(f"다운로드 완료: {downloaded}개, 이미지 없음: {skipped}건, 실패: {len(failed)}건")
    if failed:
        print(failed)
    # 화질 확인: 긴 변 기준으로 작은 것부터
    size_df = pd.DataFrame(sizes, columns=["파일", "가로", "세로"])
    size_df["긴변"] = size_df[["가로", "세로"]].max(axis=1)
    return size_df.sort_values("긴변")

size_df = download_si_images(pure_hyungbae)
size_df.head(10)